# Toxic Comment Classification on Civil Comments

**Name:** Redowanul Akbar 
**Student ID:** 1000129290 

Pipeline:
1. Data loading and labelling (`toxicity >= 0.5` means toxic)
2. Transfer learning: frozen BERT-based encoder + logistic regression (baseline)
3. Fine-tuning the whole model with default hyperparameters
4. Hyperparameter tuning with Optuna
5. Retraining with the best hyperparameters
6. Evaluation and comparison on the test set
7. Test on real-world text
8. Interpretability with SHAP

Set `PRESET = "quick"` for a small run on a CPU. The reported results use `"full"` on a GPU.

In [ ]:
%matplotlib inline
import pandas as pd
from IPython.display import Image, display

from src.config import get_config
from src import pipeline
from src.report import update_readme

pd.set_option("display.max_colwidth", 120)

PRESET = "full"   # "full" or "quick"
cfg = get_config(PRESET)
state = pipeline.init_state(cfg)
cfg.to_dict()

## 1. Data

The dataset is loaded with `load_dataset("google/civil_comments")`. A comment is labelled **toxic** when its `toxicity` score is at least **0.5**.
The training subset is sampled with 30% toxic comments; validation and test keep the natural class balance.

In [ ]:
stats = pipeline.step_data(state)
stats

In [ ]:
state["train"][["text", "toxicity", "label"]].sample(8, random_state=1)

## 2. Transfer learning: frozen encoder baseline

The pre-trained encoder is only used as a feature extractor: all of its weights are frozen and each comment is turned into a mean-pooled embedding.
A logistic regression classifier is trained on these embeddings. No hyperparameters are tuned here.

In [ ]:
pipeline.step_frozen_baseline(state)

## 3. Fine-tuning (default hyperparameters)

The whole model is unfrozen and trained end-to-end with learning rate 2e-5, batch size 16, 2 epochs, weight decay 0.01 and warmup ratio 0.1.

In [ ]:
history_default = pipeline.step_finetune_default(state)
pd.DataFrame(history_default).drop(columns="step_losses")

## 4. Hyperparameter tuning

Optuna (TPE sampler with median pruning) searches over learning rate, batch size, epochs, weight decay and warmup ratio.
The objective is validation PR-AUC, which suits the imbalanced validation set. Trial 0 is the default setting from section 3.

In [ ]:
tuning_table = pipeline.step_tuning(state)
tuning_table

## 5. Fine-tuning with the best hyperparameters

In [ ]:
history_tuned = pipeline.step_finetune_tuned(state)
pd.DataFrame(history_tuned).drop(columns="step_losses")

## 6. Evaluation: frozen baseline vs fine-tuned vs tuned

Metrics are reported at the default threshold of 0.5 and at the F1-optimal threshold chosen on the validation set.

In [ ]:
m05, mthr = pipeline.step_evaluate(state)
display(m05.round(4))
display(mthr.round(4))

## 7. Real-world text

In [ ]:
pipeline.step_real_world(state)

## 8. Interpretability with SHAP

A small, deliberately chosen set of predictions is explained: confident correct predictions, the clearest mistakes from the test set, and some real-world sentences.
Red tokens push the prediction towards **toxic**, blue tokens towards **not toxic**.

In [ ]:
explanations = pipeline.step_explain(state)
explanations[["group", "text", "predicted", "actual", "prob_toxic", "toward_toxic", "toward_not_toxic"]]

In [ ]:
for i in explanations["id"]:
    display(Image(filename=str(cfg.figures_dir / "explanations" / f"example_{i:02d}_heatmap.png")))

In [ ]:
import shap
shap.plots.text(state["shap_values"][:, :, 1])

## 9. Write the results into README.md

In [ ]:
update_readme(state)